# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields
def show_recordsets_and_fields(ds):
    print('Available Record Sets:')
    for rs in ds.metadata.record_sets:
        print(f"  RecordSet: {rs['@id']} (name: {rs.get('name','N/A')})")
        if 'field' in rs:
            fields = rs['field']
            if not isinstance(fields, list):
                fields = [fields]
            for field in fields:
                if isinstance(field, dict):
                    field_id = field.get('@id','')
                    field_name = field.get('name','')
                else:
                    field_id = field
                    field_name = ''
                print(f"    - Field: {field_id} {('('+field_name+')') if field_name else ''}")
        else:
            print('    No fields listed.')
        print()
        
show_recordsets_and_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose Record Set(s) by @id (replace with IDs from the overview as necessary)
# Let's enumerate all record sets first to pick up an ID.
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records from record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records from: {record_set}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found in record set {record_set}.")
        print()

# For further steps, pick one populated record set:
example_record_set = None
for k in dataframes:
    if len(dataframes[k]) > 0:
        example_record_set = k
        break
if example_record_set:
    print(f"Proceeding with record set: {example_record_set}")
    print('Columns:', dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())
else:
    print('No non-empty record set to use.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field if present in this record set
# See which columns look numeric
import numpy as np
df = dataframes.get(example_record_set)
numeric_field = None
group_field = None
if df is not None:
    for col in df.columns:
        # Try to convert to numeric
        try:
            df[col] = pd.to_numeric(df[col])
            if numeric_field is None:
                numeric_field = col
        except Exception:
            # Not numeric
            pass
    
    print(f"Using numeric field for analysis: {numeric_field}")
    
    # Try to find another non-numeric column for grouping
    for col in df.columns:
        if not np.issubdtype(df[col].dtype, np.number):
            group_field = col
            break
    print(f"Grouping field (if present): {group_field}")

    # Filter based on threshold
    threshold = None
    if numeric_field is not None:
        if df[numeric_field].dtype.kind in 'iufc':
            threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())
            # Normalize
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped data by {group_field} (showing means):")
                print(grouped_df.head())
        else:
            print(f"Field {numeric_field} is not numeric.")
    else:
        print("No numeric field found for EDA.")
else:
    print('No dataframe loaded. Try re-running previous cells.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if df is not None and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field or dataframe found for plotting.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Enter your observations here. Example: The dataset contains protein mass spectrometry records with abundant numeric and categorical fields for quantitative analysis and comparison across sample groups. -->